In [8]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df = pd.read_csv('Covid-19_svodnye.csv')
df

,"COVID-19 (1-переболевшие, 0-здоровые)",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,№,Дата,Пол,Фамилия,Кортизол,Глюкоза,Молочная к-та,Инсулин,МДА (Эритроциты),Эритроциты,...,NaN,NaN,NaN,МДА (Плазма),Плазма,NaN,NaN,NaN,NaN,З/П
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,ФЛ,...,НЭЖК,ТГ,ЭХС,NaN,ФЛ,СХ,НЭЖК,ТГ,ЭХС,NaN
2,3,2/11/2021,ж,Идиятова,800,3.9,1.7,0.126,23.1,23%,...,21%,18%,22%,4.9,10%,21%,28%,20%,21%,0
3,5,2/17/2021,м,Поторочин,516,7,4.9,0.162,25.5,24%,...,15%,15%,18%,5,25%,15%,24%,14%,23%,0
4,15,3/24/2021,ж,Гореева,450,7.6,6.6,0.096,25.6,21%,...,16%,20%,15%,4.6,16%,15%,33%,13%,23%,0
5,21,4/14/2021,м,Белан,936,4.6,1.7,0.168,30.5,23%,...,15%,22%,14%,4.5,13%,15%,31%,14%,26%,0
6,34,6/16/2021,м,Вахрушев,449,3.8,4.9,0.127,20.5,20%,...,17%,20%,18%,4.7,23%,12%,23%,17%,27%,0
7,NaN,NaN,м,М 38,235,4.6,1.7,0.134,28.2,21%,...,22%,17%,22%,4.5,31%,7%,25%,15%,22%,0
8,NaN,NaN,м,М 41,302,7.6,4.9,0.185,19.2,13%,...,17%,22%,24%,5,9%,14%,30%,22%,26%,0
9,NaN,NaN,м,М 49,480,3.8,6.6,0.178,27.7,40%,...,13%,17%,15%,4.8,17%,11%,32%,13%,27%,0


In [15]:
import pandas as pd

# 1. Загружаем данные, пропуская первую строку с общим заголовком
df = pd.read_csv('Covid-19_svodnye.csv', header=None, skiprows=1)

# 2. Определяем новые понятные названия столбцов[cite: 2]
new_cols = [
    "ID", "Дата", "Пол", "Фамилия", "Кортизол", "Глюкоза", "Молочная к-та", "Инсулин", 
    "МДА (Эритроциты)", "Эритроциты_ФЛ", "Эритроциты_СХ", "Эритроциты_НЭЖК", "Эритроциты_ТГ", "Эритроциты_ЭХС",
    "МДА (Плазма)", "Плазма_ФЛ", "Плазма_СХ", "Плазма_НЭЖК", "Плазма_ТГ", "Плазма_ЭХС", "Target"
]

# Убираем строки, которые были частью сложного заголовка, и назначаем новые имена
df = df.iloc[2:].copy()
df.columns = new_cols

# 3. Удаляем столбцы, которые не помогут модели найти закономерности (шум)[cite: 1]
df_clean = df.drop(columns=["ID", "Дата", "Фамилия"])

# 4. Преобразуем пол в числа: 'ж' -> 0, 'м' -> 1
df_clean['Пол'] = df_clean['Пол'].astype(str).str.strip().str.lower()
df_clean['Пол'] = df_clean['Пол'].map({'ж': 0, 'м': 1})

# 5. Очищаем все столбцы от символов '%' и переводим в числовой формат[cite: 1, 2]
for col in df_clean.columns:
    if df_clean[col].dtype == object:
        # Убираем проценты и лишние пробелы
        df_clean[col] = df_clean[col].astype(str).str.replace('%', '').str.strip()
    
    # Преобразуем в float, заменяя ошибки на NaN (если они будут)
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean

,Пол,Кортизол,Глюкоза,Молочная к-та,Инсулин,МДА (Эритроциты),Эритроциты_ФЛ,Эритроциты_СХ,Эритроциты_НЭЖК,Эритроциты_ТГ,Эритроциты_ЭХС,МДА (Плазма),Плазма_ФЛ,Плазма_СХ,Плазма_НЭЖК,Плазма_ТГ,Плазма_ЭХС,Target
2,0,800,3.9,1.7,0.126,23.1,NaN,NaN,NaN,NaN,NaN,4.9,NaN,NaN,NaN,NaN,NaN,0
3,1,516,7.0,4.9,0.162,25.5,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,0
4,0,450,7.6,6.6,0.096,25.6,NaN,NaN,NaN,NaN,NaN,4.6,NaN,NaN,NaN,NaN,NaN,0
5,1,936,4.6,1.7,0.168,30.5,NaN,NaN,NaN,NaN,NaN,4.5,NaN,NaN,NaN,NaN,NaN,0
6,1,449,3.8,4.9,0.127,20.5,NaN,NaN,NaN,NaN,NaN,4.7,NaN,NaN,NaN,NaN,NaN,0
7,1,235,4.6,1.7,0.134,28.2,NaN,NaN,NaN,NaN,NaN,4.5,NaN,NaN,NaN,NaN,NaN,0
8,1,302,7.6,4.9,0.185,19.2,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN,NaN,NaN,NaN,0
9,1,480,3.8,6.6,0.178,27.7,NaN,NaN,NaN,NaN,NaN,4.8,NaN,NaN,NaN,NaN,NaN,0
10,1,745,3.8,6.6,0.146,20.5,NaN,NaN,NaN,NaN,NaN,4.9,NaN,NaN,NaN,NaN,NaN,0
11,1,816,4.6,1.7,0.091,25.6,NaN,NaN,NaN,NaN,NaN,4.5,NaN,NaN,NaN,NaN,NaN,0


In [20]:
# Удаляет все СТОЛБЦЫ, в которых есть хотя бы одно пустое значение (NaN)
df_clean = df_clean.dropna(axis=1)

# Если нужно удалить СТРОКИ с пропусками, используйте axis=0 (или просто dropna())
df_clean = df_clean.dropna()
df_clean = df_clean.rename(columns={'Target': 'З/П'})
df_clean

,Пол,Кортизол,Глюкоза,Молочная к-та,Инсулин,МДА (Эритроциты),МДА (Плазма),З/П
2,0,800,3.9,1.7,0.126,23.1,4.9,0
3,1,516,7.0,4.9,0.162,25.5,5.0,0
4,0,450,7.6,6.6,0.096,25.6,4.6,0
5,1,936,4.6,1.7,0.168,30.5,4.5,0
6,1,449,3.8,4.9,0.127,20.5,4.7,0
7,1,235,4.6,1.7,0.134,28.2,4.5,0
8,1,302,7.6,4.9,0.185,19.2,5.0,0
9,1,480,3.8,6.6,0.178,27.7,4.8,0
10,1,745,3.8,6.6,0.146,20.5,4.9,0
11,1,816,4.6,1.7,0.091,25.6,4.5,0


In [22]:
df_clean.to_csv('df_clean_g', index= 'False')